## This is a tutorial for using the code in the file 'kuramoto_simulation.py'

First, make sure the conda environment from metronomes.yaml is created and activated. To do this run the following commands:

In [1]:
#! conda env create -f ../environments/metronomes.yaml
#! conda activate metronomes

Then create a new python or jupyter notebook, making sure that kuramoto_simulation.py is in the same directory.
To start making the simulation import the kuramoto simulation as such:

In [2]:
from kuramoto_simulation import Window, Screen_params, Model_params, Data_Collector, Standard_Step
import numpy as np

pygame 2.6.1 (SDL 2.32.56, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


### Explaining the different classes and their uses

#### Window: the timestep for the simulation

This class iterates the pygame window and handles the update at each timestep. <br>
To initialise a Window two parameters need to be defined: <br>
Screen_params and Model_params.

#### Screen_params:

The parameters for the drawing of the simulation, nothing to do with the maths, but only the visualisation.
It is a data class with the screen width, height, background colour and model radius defined:

In [3]:
# Defining Screen_params

# the width, height and radius must be ints. Background colour is an RGB value so must be a tuple of 3 ints.
screen_params = Screen_params(800, # width
                              800, # height
                              350, # radius
                              (0, 20, 80)) # background colour, RGB

#### Model_params:

The parameters for the model itself, defining the oscillators initial phases and natural frequencies as well as the update rule and coupling strength. <br>
The initial phases and natural frequencies are passed into Model_params as lists, so they can be generated by defining a function that takes in N, the number of oscillators, and returns the list of natural frequencies and initial phases.

In [4]:
# Defining the Model_params

model_params = Model_params(0.4, # K
                            [0.3, 0.5, 0.6], # the list of natural frequencies
                            [0.2, 0.46, 1.3], # the list of initial phases
                            Standard_Step) # the step function

# alternative with functions to build phases and frequencies

# example function that generates the natural frequencies
def generate_natural_frequencies(num_frequencies: int) -> list[float]:
   return np.random.uniform(0, 0.5, num_frequencies).tolist()

# example function that generates the initial angles as evenly spaced around the circle
def generate_initial_angles(num_angles: int) -> list[float]:
   return np.linspace(0, 2*np.pi, num=num_angles).tolist()


N = 15
model_params = Model_params(0.4, # K
                            generate_natural_frequencies(N), # the list of natural frequencies
                            generate_initial_angles(N), # the list of initial phases
                            Standard_Step) # the step function

#### The step function: how to implement control and coupling:

This is probably the most important thing to choose; the step function determines how the system evolves and how control is given to it.

I will first define the simple Kuramoto control and then show where control could be implemented.

In [5]:
# Building the new step - simple Kuramoto model with the possibility of control


# a generic control function that takes in the time and oscillator position and returns a control output
def control(t: float, Y: list[float] | np.ndarray) -> float:
   return 0

def new_step(t: float, Y: list[float] | np.ndarray, K: float, N: int, nat_freqs: list[float]) -> np.ndarray:
   
   # the derivatives - initialised as empty
   dYdt = []

   # then each oscillator is looped through and the updated angular velocity is added to dYdt
   for i in range(N):
      # this is where the control can be added - just added to the system response.
      d_theta_dt = nat_freqs[i] + K/N * sum(np.sin(Y[j] - Y[i]) for j in range(N)) + control(t, Y)
      dYdt.append(d_theta_dt)
   
   # the derivatives are then returned
   return np.array(dYdt)

This modified step function can then be included in the definition of the Model_params:


In [6]:
model_params = Model_params(0.4,
                            generate_natural_frequencies(N),
                            generate_initial_angles(N),
                            new_step)

### Window: where the model is built and run

The Window class takes as arguments the screen and model parameters (and an optional data collector) and then runs the simulation

In [ ]:
# To build the window just initialise everything as:

window = Window(screen_params,
                model_params)

# to run the model just run the command:

#! This model doesn't run very nicely in a notebook - it is more stable when run in a python script !#
window.main()


Oscillators:
Natural Frequency: 0.49243411722076974, Angle: 3.4873016114659734
Natural Frequency: 0.30382789097626345, Angle: 2.516717793846313
Natural Frequency: 0.40699192987844685, Angle: 3.680752514764979
Natural Frequency: 0.2711033087240006, Angle: 3.052549394934312
Natural Frequency: 0.46490962448656614, Angle: 5.326970790848115
Natural Frequency: 0.08445662654933311, Angle: 2.5266238738006996
Natural Frequency: 0.3348411992222025, Angle: 5.3373638348577295
Natural Frequency: 0.44680953544627394, Angle: 6.88011514305261
Natural Frequency: 0.28860569448228257, Angle: 6.121544017745001
Natural Frequency: 0.39549372013607437, Angle: 7.3869085819693705
Natural Frequency: 0.286651953363345, Angle: 7.0401384038971635
Natural Frequency: 0.1687458404526656, Angle: 6.611656897704189
Natural Frequency: 0.3698677702595185, Angle: 8.252369108785611
Natural Frequency: 0.20499403747622486, Angle: 7.551428677458916
Natural Frequency: 0.25636389361864953, Angle: 8.17369723592397

time elapsed:

: 

### Data_collector: how to get and save data from the model

This is a class that is defined by YOU; it is initialised when the model is run, updated at every step and then finally when exited.
Then the data can be taken from the model and turned into graphs, csv files, extra data (coupling parameters, average phases etc...)